In [1]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset, DataLoader
drive.mount('/content/drive')
import cv2
from google.colab.patches import cv2_imshow
from torch.utils.data import DataLoader
import numpy as np
import torchvision.transforms as transforms
import timm
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim



KeyboardInterrupt: 

In [2]:
ROOT = "/content/drive/MyDrive/Digital_Knee_X_ray_Images"
TRAIN_DATASET_DIR = "MedicalExpert-I"
VALIDATE_DATASET_DIR = "MedicalExpert-II"

In [3]:
class KneeXRayDataset(Dataset):

    def load_images_path(self):
        #iter each category
        # print(self.categories)
        for i, category in enumerate(self.categories):
            category_path = os.path.join(self.root, category)

            #iter file in category
            for file_path in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, file_path))
                self.labels.append(i)

    def load_image_from_path(self,imagePath):
        img_bgr = cv2.imread(imagePath)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return img_rgb
        
    def __init__(self, root, train_dataset_dir, validate_dataset_dir, transform , train=True, ):
        self.root = root
        self.transform  = transform

        # determine which dir will use (train , val)
        if train :
            self.root = os.path.join(root, train_dataset_dir)
        else:
            self.root = os.path.join(root,validate_dataset_dir)

        # get all category (0 -> 4)
        self.categories = os.listdir(self.root)
        self.image_paths = []
        self.labels = []

        # load images from dataset
        self.load_images_path()

        
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        image = self.load_image_from_path(self.image_paths[idx])
        # cv2_imshow(image)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()
        return self.transform(image), self.labels[idx]



In [4]:
import cv2

#function too add padding to image
class SquarePadOpenCV(object):
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, 
            pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, 
            value=[0, 0, 0]
        )
        return padded_image

In [5]:
transform = transforms.Compose([
    SquarePadOpenCV(),
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
train_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, transform=transform)
validate_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, train=False , transform=transform)

train_dataset_loader =  DataLoader(train_dataset, batch_size=16, shuffle=True , drop_last=False)
validate_dataset_loader = DataLoader(validate_dataset, batch_size=16, drop_last=False)

In [7]:
#load avaiable device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
#define transfer learning model function
def create_transfer_learning_model(model_name: str="densenet121", pretrained: bool = True, num_classes:int = 5, device = device):
    model = timm.create_model(model_name, pretrained= pretrained, num_classes=5)

    # #frezze all other layer
    # for param in model.parameters():
    #     param.requires_grad = False

    # #unfrezze the class classifier10
    # for param in model.classifier.parameters():
    #     param.requires_grad = True

    #unlock all layer
    for param in model.parameters():
        param.requires_grad = True

    return model.to(device)


densenet121_model = create_transfer_learning_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, densenet121_model.parameters()), lr=1e-5)


In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    # model is tranining mode
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc=" Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # delete previous grad
        optimizer.zero_grad()
        outputs = model(images)

        # calc loss
        loss = criterion(outputs, labels)

        # backward to calc loss
        loss.backward()

        # optimize model
        optimizer.step()

        # result
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total

In [11]:
def validate(model, loader, criterion, device):

    # eval mode
    model.eval()

    # init
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=" Validating", leave=False):

            #load to device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total 

    

In [12]:
EPOCHS = 15
best_val_acc = 0.0
save_path = '/content/drive/MyDrive/AI/models/DenseNet121/dense_net_121_transfer.pth'
parent_dir = os.path.dirname(save_path)
os.makedirs(parent_dir, exist_ok=True)


for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(densenet121_model, train_dataset_loader, criterion, optimizer, device)

    print(f"  -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    val_loss, val_acc = validate(densenet121_model, validate_dataset_loader, criterion, device)
    print(f"  -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        torch.save(densenet121_model.state_dict(), save_path)

        print(f"  [Save] Best val acc: {best_val_acc:.2f}%")
        


  -> Train Loss: 1.5973 | Train Acc: 25.45%


  -> Val Loss: 1.4089 | Val Acc: 50.24%
  [Save] Best val acc: 50.24%


  -> Train Loss: 1.3356 | Train Acc: 52.79%


  -> Val Loss: 1.2638 | Val Acc: 53.82%
  [Save] Best val acc: 53.82%


  -> Train Loss: 1.1701 | Train Acc: 61.70%


...

  -> Val Loss: 0.2891 | Val Acc: 96.73%
  [Save] Best val acc: 96.73%


  -> Train Loss: 0.3519 | Train Acc: 93.39%


  -> Val Loss: 0.2679 | Val Acc: 97.15%
  [Save] Best val acc: 97.15%


  -> Train Loss: 0.3203 | Train Acc: 94.00%


  -> Val Loss: 0.2345 | Val Acc: 98.42%
  [Save] Best val acc: 98.42%


In [13]:
def predict_single_image_opencv(image_path, model, device):
    img_bgr = cv2.imread(image_path)
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    img_tensor = transform(img_rgb)
    
    img_batch = img_tensor.unsqueeze(0)
    
    img_batch = img_batch.to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_batch)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)
        
    categories = ['0Normal', '1Doubtful', '2Mild', '3Moderate', '4Severe']
    return categories[predicted_idx.item()], confidence.item() * 100

In [14]:
def load_model_from_weight_file(model, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint)

    model.eval()

    return model.to(device)

In [15]:
model = timm.create_model('densenet121', pretrained=False, num_classes=5)
model = load_model_from_weight_file(model, save_path, device)

In [16]:
image_test = "/content/drive/MyDrive/Digital_Knee_X_ray_Images/MedicalExpert-II/4Severe/SevereG4 (206).png"
predicted_class, confidence_score = predict_single_image_opencv(image_test, model, device)
print(predicted_class)
print(confidence_score)

4Severe
87.27701902389526
